![JohnSnowLabs](https://sparknlp.org/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/spark-nlp/blob/master/examples/python/annotation/text/english/text-similarity/PairwiseVectorSimilarity.ipynb)

# PairwiseVectorSimilarity

`PairwiseVectorSimilarity` is a Spark NLP annotator that scores every pair of sentence
embeddings across two input columns. Given N embeddings in column A and M embeddings in
column B, it computes all N x M scores in a single `transform()` call and returns one
`VECTOR_SIMILARITY` annotation per pair.

**Typical use cases**

| Pattern | Setup |
|---|---|
| Query vs corpus (retrieval) | CrossJoin a query DataFrame with a corpus DataFrame, then transform |
| Sentence-level document pairs | Use a sentence-split pipeline; the annotator scores all sentence cross-pairs per row |
| Two-stage retrieval | BM25 narrows the candidate set; vector similarity re-ranks them |

**Similarity methods**

| method | range | higher means |
|---|---|---|
| `cosine` (default) | -1.0 to 1.0 | more similar |
| `dotProduct` | unbounded | more similar |
| `euclidean` | (-inf, 0.0] | more similar (0.0 = identical) |

The `euclidean` method returns the negative L2 distance so that higher is always better
regardless of which method you choose.

This notebook covers:
1. Embedding a query set and a document corpus with a pretrained model
2. Scoring all query-document pairs and ranking results per query
3. Comparing the three similarity methods on the same data
4. Scoring sentence-level pairs (multiple embeddings per row)
5. Two-stage retrieval: BM25 candidate filtering followed by vector re-ranking
6. Inspecting the full annotation output and metadata fields

In [ ]:
# Only run this cell when you are using Spark NLP on Google Colab
!wget http://setup.johnsnowlabs.com/colab.sh -O - | bash

In [ ]:
import sparknlp
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import (
    BM25Approach, E5Embeddings, PairwiseVectorSimilarity,
    SentenceDetector, StopWordsCleaner, Tokenizer,
)
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = sparknlp.start()

print("Spark NLP version :", sparknlp.version())
print("Apache Spark version:", spark.version)

## 1. Sample data

A corpus of twelve sentences spanning four topics (nutrition, machine learning, travel,
astronomy) and four queries, one per topic.
This diversity makes it easy to see whether the annotator retrieves topically relevant
documents at the top of the ranking.

In [2]:
corpus = spark.createDataFrame([
    (1,  "Apples and oranges are excellent sources of vitamin C."),
    (2,  "Bananas are rich in potassium, which supports heart and muscle function."),
    (3,  "A diet high in fiber helps regulate blood sugar and cholesterol levels."),
    (4,  "Machine learning is a branch of artificial intelligence that learns from data."),
    (5,  "Deep learning uses multi-layer neural networks to model complex patterns."),
    (6,  "Transformers are a neural network architecture that rely on self-attention."),
    (7,  "Florence is a city in Tuscany known for Renaissance art and architecture."),
    (8,  "The Amalfi Coast in southern Italy is one of Europe's most scenic drives."),
    (9,  "Kyoto offers ancient temples, bamboo forests, and traditional tea ceremonies."),
    (10, "The James Webb Space Telescope observes the universe in infrared light."),
    (11, "Black holes form when massive stars collapse under their own gravity."),
    (12, "The Milky Way galaxy contains an estimated 100 to 400 billion stars."),
], ["id", "text"])

queries = spark.createDataFrame([
    ("q1", "Which fruits and foods are high in vitamins?"),
    ("q2", "How do neural networks process information?"),
    ("q3", "Best destinations for cultural tourism in Europe."),
    ("q4", "Recent discoveries about stars and galaxies."),
], ["qid", "text"])

print("Corpus:")
corpus.show(truncate=75)
print("Queries:")
queries.show(truncate=75)

Corpus:
+---+---------------------------------------------------------------------------+
| id|                                                                       text|
+---+---------------------------------------------------------------------------+
|  1|                     Apples and oranges are excellent sources of vitamin C.|
|  2|   Bananas are rich in potassium, which supports heart and muscle function.|
|  3|    A diet high in fiber helps regulate blood sugar and cholesterol levels.|
|  4|Machine learning is a branch of artificial intelligence that learns from...|
|  5|  Deep learning uses multi-layer neural networks to model complex patterns.|
|  6|Transformers are a neural network architecture that rely on self-attention.|
|  7|  Florence is a city in Tuscany known for Renaissance art and architecture.|
|  8|  The Amalfi Coast in southern Italy is one of Europe's most scenic drives.|
|  9|Kyoto offers ancient temples, bamboo forests, and traditional tea ceremo...|
| 10|   

## 2. Embedding pipeline

`e5_small_v2` is a 33 MB sentence embedding model that produces 384-dimensional
`SENTENCE_EMBEDDINGS` directly from a `DOCUMENT` input — no tokenizer or pooling stage
required. The same fitted pipeline is reused to embed both the corpus and the queries.

In [3]:
doc_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

embeddings = E5Embeddings.pretrained("e5_small_v2", "en") \
    .setInputCols(["document"]) \
    .setOutputCol("embeddings")

emb_pipeline = Pipeline(stages=[doc_assembler, embeddings])
emb_model = emb_pipeline.fit(corpus)

corpus_emb = emb_model.transform(corpus).select(
    F.col("id"), F.col("text"), F.col("embeddings").alias("doc_emb")
)
query_emb = emb_model.transform(queries).select(
    F.col("qid"), F.col("text").alias("query_text"), F.col("embeddings").alias("query_emb")
)

dim = len(corpus_emb.select(F.explode("doc_emb").alias("e")).select("e.embeddings").first()[0])
print(f"Embedding dimension: {dim}")

e5_small_v2 download started this may take some time.
Approximate size to download 76.2 MB
[OK!]
Embedding dimension: 384


## 3. Scoring all query-document pairs

`PairwiseVectorSimilarity` processes one row at a time, so we `crossJoin` the query and
corpus DataFrames first. With 4 queries and 12 documents this produces 48 rows.
Each row gets one output annotation holding the cosine similarity score.

In [4]:
paired = query_emb.crossJoin(corpus_emb)
print(f"Rows after crossJoin: {paired.count()}  (4 queries x 12 docs)")

pvs = PairwiseVectorSimilarity() \
    .setInputCols(["query_emb", "doc_emb"]) \
    .setOutputCol("similarity") \
    .setSimilarityMethod("cosine")

scored = (
    pvs.transform(paired)
    .select(
        F.col("qid"),
        F.col("query_text"),
        F.col("id"),
        F.col("text").alias("doc_text"),
        F.explode(F.col("similarity")).alias("s"),
    )
    .select(
        F.col("qid"),
        F.col("query_text"),
        F.col("id"),
        F.col("doc_text"),
        F.round(F.col("s.result").cast("double"), 4).alias("cosine"),
    )
    .cache()
)

print(f"Total scored pairs: {scored.count()}")
scored.orderBy(F.desc("cosine")).show(8, truncate=60)

Rows after crossJoin: 48  (4 queries x 12 docs)
Total scored pairs: 48
+---+--------------------------------------------+---+------------------------------------------------------------+------+
|qid|                                  query_text| id|                                                    doc_text|cosine|
+---+--------------------------------------------+---+------------------------------------------------------------+------+
| q1|Which fruits and foods are high in vitamins?|  1|      Apples and oranges are excellent sources of vitamin C.| 0.865|
| q2| How do neural networks process information?|  5|Deep learning uses multi-layer neural networks to model c...|0.8623|
| q4|Recent discoveries about stars and galaxies.| 10|The James Webb Space Telescope observes the universe in i...|0.8514|
| q4|Recent discoveries about stars and galaxies.| 11|Black holes form when massive stars collapse under their ...|0.8486|
| q2| How do neural networks process information?|  6|Transformers a

## 4. Top-3 documents per query

A window function ranks scored pairs within each query group so we can retrieve the
top-K documents in a single pass without a per-query sort.
The results show whether the annotator correctly groups each query with its topic.

In [5]:
window = Window.partitionBy("qid").orderBy(F.desc("cosine"))

(
    scored
    .withColumn("rank", F.rank().over(window))
    .filter(F.col("rank") <= 3)
    .orderBy("qid", "rank")
    .select("qid", "query_text", "rank", "id", "doc_text", "cosine")
    .show(truncate=55)
)

+---+-------------------------------------------------+----+---+-------------------------------------------------------+------+
|qid|                                       query_text|rank| id|                                               doc_text|cosine|
+---+-------------------------------------------------+----+---+-------------------------------------------------------+------+
| q1|     Which fruits and foods are high in vitamins?|   1|  1| Apples and oranges are excellent sources of vitamin C.| 0.865|
| q1|     Which fruits and foods are high in vitamins?|   2|  3|A diet high in fiber helps regulate blood sugar and ...|0.8317|
| q1|     Which fruits and foods are high in vitamins?|   3|  2|Bananas are rich in potassium, which supports heart ...|0.8298|
| q2|      How do neural networks process information?|   1|  5|Deep learning uses multi-layer neural networks to mo...|0.8623|
| q2|      How do neural networks process information?|   2|  6|Transformers are a neural network archit

## 5. Comparing similarity methods

All three methods score the same (query, document) pairs.
The relative ranking order is usually consistent across methods, but the score scales
differ significantly.

- `cosine` is bounded in [-1.0, 1.0] and scale-invariant (only vector direction matters).
- `dotProduct` is unbounded and sensitive to vector magnitude; scores depend on the model.
- `euclidean` returns the negative L2 distance: 0.0 means identical vectors, more negative
  means less similar.

Note: `e5_small_v2` produces L2-normalized (unit) embeddings, so for this model
`dotProduct` and `cosine` will produce identical scores. On models that do not normalize
their output vectors the two methods will diverge.

In [6]:
def score_with_method(method):
    return (
        PairwiseVectorSimilarity()
        .setInputCols(["query_emb", "doc_emb"])
        .setOutputCol("similarity")
        .setSimilarityMethod(method)
        .transform(paired)
        .select(
            F.col("qid"),
            F.col("id"),
            F.col("text").alias("doc_text"),
            F.explode(F.col("similarity")).alias("s"),
        )
        .select(
            F.col("qid"),
            F.col("id"),
            F.col("doc_text"),
            F.round(F.col("s.result").cast("double"), 4).alias(method),
        )
    )

cosine_df    = score_with_method("cosine")
dot_df       = score_with_method("dotProduct")
euclidean_df = score_with_method("euclidean")

# Join all three on the same (qid, id) key and filter to one query for readability.
comparison = (
    cosine_df
    .join(dot_df.select("qid", "id", "dotProduct"), on=["qid", "id"])
    .join(euclidean_df.select("qid", "id", "euclidean"), on=["qid", "id"])
    .filter(F.col("qid") == "q2")
    .orderBy(F.desc("cosine"))
    .select("qid", "id", "doc_text", "cosine", "dotProduct", "euclidean")
)

print("Method comparison for q2 — 'How do neural networks process information?'")
comparison.show(truncate=48)

Method comparison for q2 — 'How do neural networks process information?'
+---+---+------------------------------------------------+------+----------+---------+
|qid| id|                                        doc_text|cosine|dotProduct|euclidean|
+---+---+------------------------------------------------+------+----------+---------+
| q2|  5|Deep learning uses multi-layer neural network...|0.8623|    0.8623|  -0.5248|
| q2|  6|Transformers are a neural network architectur...|0.8457|    0.8457|  -0.5556|
| q2|  4|Machine learning is a branch of artificial in...|0.8425|    0.8425|  -0.5612|
| q2| 11|Black holes form when massive stars collapse ...| 0.821|     0.821|  -0.5984|
| q2| 10|The James Webb Space Telescope observes the u...|0.7958|    0.7958|   -0.639|
| q2|  3|A diet high in fiber helps regulate blood sug...|0.7816|    0.7816|  -0.6608|
| q2|  2|Bananas are rich in potassium, which supports...|0.7717|    0.7717|  -0.6757|
| q2|  1|Apples and oranges are excellent sources of v...

## 6. Multiple sentences per row (N x M pairs)

When each input column contains more than one embedding per row — for example when using
a `SentenceDetector` pipeline — the annotator scores **all N x M cross-pairs** and emits
one annotation per pair. This is useful for fine-grained passage-level matching.

In this example, document A has 3 sentences and document B has 2 sentences.
The annotator produces 3 x 2 = 6 output annotations, each carrying the pair indices
in its metadata so the results can be matched back to specific sentence combinations.

In [7]:
sent_assembler = DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

sentence_detector = SentenceDetector() \
    .setInputCols(["document"]) \
    .setOutputCol("sentence")

sent_embeddings = E5Embeddings.pretrained("e5_small_v2", "en") \
    .setInputCols(["sentence"]) \
    .setOutputCol("sent_emb")

sent_pipeline = Pipeline(stages=[sent_assembler, sentence_detector, sent_embeddings])

# Document A: 3 sentences about vitamins
doc_a = spark.createDataFrame([(
    "Vitamin C is found in citrus fruits. "
    "It supports immune function and collagen synthesis. "
    "A daily intake of 65 to 90 milligrams is recommended for adults.",
)], ["text"])

# Document B: 2 sentences about minerals
doc_b = spark.createDataFrame([(
    "Potassium is an essential electrolyte for heart and muscle function. "
    "Bananas, leafy greens, and beans are excellent dietary sources.",
)], ["text"])

sent_model = sent_pipeline.fit(doc_a)
emb_a = sent_model.transform(doc_a).select(F.col("sent_emb").alias("emb_a"))
emb_b = sent_model.transform(doc_b).select(F.col("sent_emb").alias("emb_b"))

sent_pvs = PairwiseVectorSimilarity() \
    .setInputCols(["emb_a", "emb_b"]) \
    .setOutputCol("sim") \
    .setSimilarityMethod("cosine")

sent_result = (
    sent_pvs.transform(emb_a.crossJoin(emb_b))
    .select(F.explode(F.col("sim")).alias("s"))
    .select(
        F.col("s.metadata")["sentence_a_idx"].cast("int").alias("sent_a"),
        F.col("s.metadata")["sentence_b_idx"].cast("int").alias("sent_b"),
        F.col("s.metadata")["sentence_a_text"].alias("text_a"),
        F.col("s.metadata")["sentence_b_text"].alias("text_b"),
        F.round(F.col("s.result").cast("double"), 4).alias("cosine"),
    )
    .orderBy("sent_a", "sent_b")
)

print(f"Output annotation count: {sent_result.count()}  (3 sentences x 2 sentences = 6 pairs)")
sent_result.show(truncate=52)

e5_small_v2 download started this may take some time.
Approximate size to download 76.2 MB
[OK!]
Output annotation count: 6  (3 sentences x 2 sentences = 6 pairs)
+------+------+----------------------------------------------------+----------------------------------------------------+------+
|sent_a|sent_b|                                              text_a|                                              text_b|cosine|
+------+------+----------------------------------------------------+----------------------------------------------------+------+
|     0|     0|                Vitamin C is found in citrus fruits.|Potassium is an essential electrolyte for heart a...|0.8102|
|     0|     1|                Vitamin C is found in citrus fruits.|Bananas, leafy greens, and beans are excellent di...|0.8285|
|     1|     0| It supports immune function and collagen synthesis.|Potassium is an essential electrolyte for heart a...|0.8398|
|     1|     1| It supports immune function and collagen synthe

## 7. Two-stage retrieval: BM25 + vector re-ranking

Running vector similarity over an entire corpus scales as O(Q x D), which becomes
expensive for large D. A common production pattern is to use a fast lexical retriever
as a first stage to narrow the candidate set to a few hundred documents, then re-rank
only those candidates with the slower but more accurate vector similarity.

Here we:
1. Train `BM25Approach` on the corpus to build a term-frequency index.
2. Run a query to retrieve only documents that share at least one term with the query.
3. Embed those candidates and re-rank them with `PairwiseVectorSimilarity`.

Comparing `bm25_score` and `vector_score` in the output shows where the two methods agree
and where vector similarity corrects lexical mismatches.

In [8]:
bm25_pipeline = Pipeline(stages=[
    DocumentAssembler().setInputCol("text").setOutputCol("document"),
    Tokenizer().setInputCols(["document"]).setOutputCol("token"),
    StopWordsCleaner().setInputCols(["token"]).setOutputCol("clean").setCaseSensitive(False),
    BM25Approach().setInputCols(["clean"]).setOutputCol("bm25").setMinDocFreq(1),
]).fit(corpus)

# Use words that appear verbatim in the corpus so BM25 returns multiple candidates.
bm25_pipeline.stages[-1].setQuery("vitamin potassium apples bananas fiber")

candidates = (
    bm25_pipeline.transform(corpus)
    .select(F.col("id"), F.col("text"), F.explode(F.col("bm25")).alias("b"))
    .select(
        F.col("id"),
        F.col("text"),
        F.round(F.col("b.metadata")["bm25_score"].cast("double"), 4).alias("bm25_score"),
    )
    .filter(F.col("bm25_score") > 0)
    .orderBy(F.desc("bm25_score"))
)

print(f"BM25 candidates: {candidates.count()} of {corpus.count()} documents passed the filter")
candidates.show(truncate=72)

BM25 candidates: 3 of 12 documents passed the filter
+---+------------------------------------------------------------------------+----------+
| id|                                                                    text|bm25_score|
+---+------------------------------------------------------------------------+----------+
|  1|                  Apples and oranges are excellent sources of vitamin C.|    4.7354|
|  2|Bananas are rich in potassium, which supports heart and muscle function.|    4.3025|
|  3| A diet high in fiber helps regulate blood sugar and cholesterol levels.|    2.0572|
+---+------------------------------------------------------------------------+----------+



In [9]:
candidates_emb = emb_model.transform(candidates).select(
    "id", "text", "bm25_score", F.col("embeddings").alias("doc_emb")
)

rerank_query = spark.createDataFrame(
    [("q1", "Which fruits and foods are high in vitamins?")], ["qid", "text"]
)
rerank_query_emb = emb_model.transform(rerank_query).select(
    "qid", F.col("text").alias("query_text"), F.col("embeddings").alias("query_emb")
)

print("Re-ranked candidates (ordered by vector similarity):")
(
    pvs.transform(rerank_query_emb.crossJoin(candidates_emb))
    .select(
        F.col("query_text"),
        F.col("id"),
        F.col("text").alias("doc_text"),
        F.col("bm25_score"),
        F.explode(F.col("similarity")).alias("s"),
    )
    .select(
        F.col("query_text"),
        F.col("id"),
        F.col("doc_text"),
        F.col("bm25_score"),
        F.round(F.col("s.result").cast("double"), 4).alias("vector_score"),
    )
    .orderBy(F.desc("vector_score"))
    .show(truncate=55)
)

Re-ranked candidates (ordered by vector similarity):
+--------------------------------------------+---+-------------------------------------------------------+----------+------------+
|                                  query_text| id|                                               doc_text|bm25_score|vector_score|
+--------------------------------------------+---+-------------------------------------------------------+----------+------------+
|Which fruits and foods are high in vitamins?|  1| Apples and oranges are excellent sources of vitamin C.|    4.7354|       0.865|
|Which fruits and foods are high in vitamins?|  3|A diet high in fiber helps regulate blood sugar and ...|    2.0572|      0.8317|
|Which fruits and foods are high in vitamins?|  2|Bananas are rich in potassium, which supports heart ...|    4.3025|      0.8298|
+--------------------------------------------+---+-------------------------------------------------------+----------+------------+



## 8. Inspecting annotation output

Each `VECTOR_SIMILARITY` annotation is a standard Spark NLP struct. Its `metadata` map
contains all information needed to identify the pair and its score without any secondary
join back to the source DataFrames.

In [10]:
# Show the full annotation struct for one specific (query, document) pair.
print("Full annotation struct (q1 vs document 1):")
(
    pvs.transform(paired)
    .filter(F.col("qid") == "q1")
    .filter(F.col("id") == 1)
    .select(F.explode(F.col("similarity")).alias("s"))
    .select(
        F.col("s.annotatorType"),
        F.col("s.begin"),
        F.col("s.end"),
        F.col("s.result"),
        F.col("s.metadata"),
    )
    .show(truncate=False)
)

# Unpack each metadata field individually for easy downstream access.
print("Metadata fields (unpacked):")
(
    pvs.transform(paired)
    .filter(F.col("qid") == "q1")
    .filter(F.col("id") == 1)
    .select(F.explode(F.col("similarity")).alias("s"))
    .select(
        F.col("s.metadata")["sentence_a_idx"].alias("a_idx"),
        F.col("s.metadata")["sentence_b_idx"].alias("b_idx"),
        F.col("s.metadata")["sentence_a_text"].alias("query"),
        F.col("s.metadata")["sentence_b_text"].alias("document"),
        F.col("s.metadata")["similarity"].cast("double").alias("score"),
        F.col("s.metadata")["similarityMethod"].alias("method"),
    )
    .show(truncate=False)
)

Full annotation struct (q1 vs document 1):
+-----------------+-----+---+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|annotatorType    |begin|end|result            |metadata                                                                                                                                                                                                                                            |
+-----------------+-----+---+------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|vector_similarity|0    |0  |0.8650460961445957|{sentence_b_idx -> 0, simil

## Notes

**Choosing a similarity method**

- Use `cosine` (default) when comparing embeddings from most pretrained models.
  Cosine similarity is scale-invariant — only the direction of the vector matters,
  not its magnitude — which makes it robust across different model families.
- Use `dotProduct` when the model was trained with a dot-product objective
  (e.g. bi-encoders trained with in-batch negatives). In that case the vector magnitude
  carries a confidence signal that cosine would discard.
- Use `euclidean` when you need a proper distance metric. The annotator returns the
  negative L2 distance so that higher scores still mean more similar, consistent with
  the other two methods.

**LightPipeline is not supported**

`LightPipeline` merges all input columns into a single flat list, making it impossible
to distinguish column A embeddings from column B embeddings. Always call `transform()`
on a Spark DataFrame.

**Scaling to large corpora**

A raw `crossJoin` grows as O(Q x D): 1,000 queries against 1,000,000 documents produces
one billion rows. For large corpora, use a first-stage filter (BM25, locality-sensitive
hashing, or an ANN index) to reduce D to a few hundred candidates per query before
applying `PairwiseVectorSimilarity`, as shown in section 7.